# Civic complaint image classifier: train in Colab

Runs the whole pipeline: download Kaggle datasets, inspect them, merge into one dataset,
train YOLO classification, evaluate, save the model.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.
Cells marked **EDIT** need your input.

## 1. Check the GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU: switch the runtime to T4 GPU, otherwise training will be very slow.")

## 2. Get the project code and install packages

In [ ]:
import os
from google.colab import userdata

# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left sidebar).
try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/deepshikhathakur020105/AI_civic_managemnet_system.git"
except Exception:
    url = "https://github.com/deepshikhathakur020105/AI_civic_managemnet_system.git"

if not os.path.exists("/content/civic"):
    !git clone -q -b ml-dataset-yolo {url} /content/civic
%cd /content/civic/ml
!pip install -q ultralytics pyyaml scikit-learn kagglehub

## 3. Kaggle login
In Kaggle: Settings > API > **Create New Token**. In Colab, open the key icon (Secrets) in the left sidebar
and add the secret(s), with notebook access turned on. Never paste the token into a cell or share it in chat.

- New-style token (starts with `KGAT_`): one secret named `KAGGLE_API_TOKEN`.
- Legacy `kaggle.json`: two secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY`.

In [ ]:
import os
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if secret("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = secret("KAGGLE_API_TOKEN")
    print("Loaded KAGGLE_API_TOKEN")
elif secret("KAGGLE_USERNAME") and secret("KAGGLE_KEY"):
    os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY")
    print("Loaded legacy username/key")
else:
    # No secret set: ask for the token in a hidden prompt (kept for this session only, never saved to the file)
    import getpass
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Paste your Kaggle API token (starts with KGAT_): ")
    print("Token loaded for this session")

# If downloads fail with an auth error, update the client first: !pip install -U kagglehub kaggle

## 4. Download the datasets  (**EDIT**)
Put one Kaggle dataset slug per class group. The slug is the `owner/name` part of the dataset URL.
Use only datasets that contain **images** (check the Data tab on Kaggle: jpg/png files).
CSV-only datasets (sensor readings, fault tables) cannot train a photo classifier.

In [ ]:
import kagglehub

DATASETS = {
    # name you choose : kaggle slug
    "garbage": "sumn2u/garbage-classification-v2",
    # "pothole": "<owner>/<pothole-dataset-slug>",       # EDIT: add an image dataset
    # "water_leakage": "<owner>/<leak-images-slug>",      # EDIT: must contain images
    # "streetlight": "<owner>/<broken-streetlight-images-slug>",
}

PATHS = {}
for name, slug in DATASETS.items():
    PATHS[name] = kagglehub.dataset_download(slug)
    print(name, "->", PATHS[name])

## 5. Inspect what you downloaded
This shows the real folder names and how many images each has. You need these names for step 6.
If a dataset shows **0 images**, it is not usable for this model.

In [ ]:
from pathlib import Path
from collections import Counter

IMG = {".jpg", ".jpeg", ".png"}

def inspect_ds(root, depth=2):
    root = Path(root)
    exts = Counter(p.suffix.lower() for p in root.rglob("*") if p.is_file())
    n_img = sum(c for e, c in exts.items() if e in IMG)
    print(f"{root}\n  file types: {dict(exts.most_common(6))}\n  images: {n_img}")
    if n_img == 0:
        print("  >>> NO IMAGES: not usable for the photo classifier")
        return
    # image count per folder that directly contains images
    per_dir = Counter(p.parent.relative_to(root) for p in root.rglob("*") if p.suffix.lower() in IMG)
    for d, c in sorted(per_dir.items())[:40]:
        print(f"    {str(d):55s} {c}")
    has_labels = any("labels" in p.parts for p in root.rglob("*.txt"))
    print("  YOLO label files found:", has_labels)

for name, path in PATHS.items():
    print("=" * 70, "\n", name)
    inspect_ds(path)

## 6. Map folders to your civic classes  (**EDIT**)
Each source is either:
- `folders`: images sorted into folders. Map **folder name -> your class**.
- `yolo`: a detection dataset with `images/` and `labels/`. Give its class names (from its `data.yaml`) in order and map **its class name -> your class**.

Use the folder names printed in step 5. Folders you leave out of `map` are ignored.
Include an `other` class of ordinary scenes with no problem (clean road, working lamp), otherwise the model
must force every photo into a civic class.

The example below maps all waste-type folders to `garbage`. Change it if your dataset's folders differ.

In [ ]:
import yaml

config = {
    "classes": ["pothole", "garbage", "water_leakage", "broken_streetlight", "other"],
    "sources": [
        {
            "name": "garbage",
            "type": "folders",
            "path": PATHS["garbage"],
            "map": {   # EDIT: folder name -> class (names must match step 5 exactly)
                "plastic": "garbage", "paper": "garbage", "cardboard": "garbage",
                "glass": "garbage", "metal": "garbage", "trash": "garbage",
                "biological": "garbage", "clothes": "garbage", "shoes": "garbage", "battery": "garbage",
            },
        },
        # {
        #     "name": "pothole", "type": "yolo", "path": PATHS["pothole"],
        #     "names": ["pothole"], "map": {"pothole": "pothole"},
        # },
        # {
        #     "name": "pothole_folders", "type": "folders", "path": PATHS["pothole"],
        #     "map": {"Pothole": "pothole", "Normal": "other"},
        # },
    ],
}
yaml.safe_dump(config, open("datasets.yaml", "w"))
print(open("datasets.yaml").read())

## 7. Merge, de-duplicate and split into train / val / test
Read the counts and warnings it prints. A class with fewer than about 200 images will be weak.
`--max-per-class` stops one big dataset from drowning the small ones.

In [ ]:
!python prepare_dataset.py --config datasets.yaml --out data/civic --max-per-class 2000

## 8. Train
Starts from a pretrained model and fine-tunes it. Watch `top1_acc` on the validation set rise.
Use fewer `--epochs` for a quick first run (e.g. 5), then 30 for the real one.

In [ ]:
!python train.py --data data/civic --epochs 30 --device 0

## 9. Evaluate on the held-out test set
Writes `reports/metrics.json` and `reports/confusion_matrix.csv`. Use these numbers in your project report,
not the training accuracy.

In [ ]:
!python evaluate.py --data data/civic
import pandas as pd
display(pd.read_csv("reports/confusion_matrix.csv"))

## 10. Try it on one image
Upload a photo in the left Files panel, then set the path below.

In [ ]:
from ultralytics import YOLO
model = YOLO("../backend/models/best.pt")
r = model("/content/test.jpg", verbose=False)[0]   # EDIT: path of your uploaded photo
top = r.probs.top5
print([(r.names[i], round(float(r.probs.data[i]), 3)) for i in top])

## 11. Save the model
Colab storage is wiped when the session ends, so download it now and copy it to Google Drive.
Put the file in `backend/models/best.pt` in the project (it is git-ignored, do not commit it).

In [ ]:
from google.colab import files, drive
import shutil, os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/civic_model", exist_ok=True)
shutil.copy("../backend/models/best.pt", "/content/drive/MyDrive/civic_model/best.pt")
shutil.copytree("reports", "/content/drive/MyDrive/civic_model/reports", dirs_exist_ok=True)
files.download("../backend/models/best.pt")